# Solution: produce_01 — your first producer

In [ ]:
from confluent_kafka import Producer
import json, time

BROKER = 'redpanda:29092'

producer = Producer({'bootstrap.servers': BROKER, 'client.id': 'python-producer'})
print(f'Producer connected to {BROKER}')

In [ ]:
topic_name    = 'strom'
message_key   = 'haus_a'
message_value = json.dumps({
    'sensor':    'strom',
    'haus':      'haus_a',
    'wert':      42.5,
    'einheit':   'kWh',
    'timestamp': time.time(),
})
print(message_value)

In [ ]:
def delivery_report(err, msg):
    if err:
        print(f'Delivery failed: {err}')
    else:
        print(f'Delivered to {msg.topic()} [{msg.partition()}] offset {msg.offset()}')

producer.produce(
    topic_name,
    key=message_key.encode('utf-8'),
    value=message_value.encode('utf-8'),
    callback=delivery_report,
)
producer.flush()   # block until the broker confirms delivery

## Task A — second key

In [ ]:
value_b = json.dumps({
    'sensor': 'strom', 'haus': 'haus_b', 'wert': 38.1, 'einheit': 'kWh',
    'timestamp': time.time(),
})
producer.produce('strom', key=b'haus_b', value=value_b.encode(), callback=delivery_report)
producer.flush()

## Task B — different topic

In [ ]:
value_w = json.dumps({
    'sensor': 'wasser', 'haus': 'haus_a', 'wert': 120.5, 'einheit': 'Liter',
    'timestamp': time.time(),
})
producer.produce('wasser', key=b'haus_a', value=value_w.encode(), callback=delivery_report)
producer.flush()

## Task C — broken broker

In [ ]:
errors = []
def error_cb(err, msg):
    errors.append(str(err))

p_broken = Producer({
    'bootstrap.servers':  'localhost:9999',
    'message.timeout.ms': 4000,
    'socket.timeout.ms':  2000,
})
p_broken.produce('strom', key=b'haus_a', value=b'{"test": true}', callback=error_cb)
p_broken.flush(timeout=5)
print(errors[0] if errors else 'unexpectedly delivered')